In [2]:
import pandas as pd
import numpy as np
from datetime import datetime
import os
import threading
import time

# Tratamento dados brutos

In [3]:
df = pd.read_csv("E-commerce Website Logs.csv",na_values=["--"], dtype={"age": "Int64"})
df['data'] = pd.to_datetime(df["accessed_date"], errors="coerce").dt.date
df.head()


,accessed_date,duration_(secs),network_protocol,ip,bytes,accessed_Ffom,age,gender,country,membership,language,sales,returned,returned_amount,pay_method,data
0,2017-03-14 17:43:57.172,2533,TCP,1.10.195.126,20100,Chrome,28,Female,CA,Normal,English,261.9600,No,0.0,Credit Card,2017-03-14
1,2017-03-14 17:43:57.172,4034,TCP,1.1.217.211,20500,Mozilla Firefox,21,Male,AR,Normal,English,731.9400,No,0.0,Debit Card,2017-03-14
2,2017-03-14 17:43:26.135,1525,TCP,1.115.198.107,90100,Mozilla Firefox,20,Male,PL,Normal,English,14.6200,No,0.0,Cash,2017-03-14
3,2017-03-14 17:43:26.135,4572,TCP,1.121.152.143,100300,Mozilla Firefox,66,Female,IN,Normal,Spanish,957.5775,No,0.0,Credit Card,2017-03-14
4,2017-03-14 18:17:09.005,3652,TCP,1.123.135.213,270200,Mozilla Firefox,53,Female,KR,Normal,Spanish,22.3680,No,0.0,Cash,2017-03-14


In [4]:
# Particionando os dados por data - Parte 1
for data in df['data'].unique():
    df_data = df[df['data'] == data]
    df_data.to_csv(f"dados_brutos/{data}.csv", index=False)

# Funções <span style="color: #0077ff">**Camada de Lote**

In [5]:
def inicializar_estado(datas):
    if not os.path.exists("gerenciamento/log_processamento.csv"):
        df_estado = pd.DataFrame({
            "data": datas,
            "status": "pendente",
            "ultima_execucao": None,
            "erro": None
        })
        df_estado.to_csv("gerenciamento/log_processamento.csv", index=False)

def atualizar_estado(data, status, erro=None):
    df_estado = pd.read_csv("gerenciamento/log_processamento.csv")

    df_estado.loc[df_estado["data"] == str(data), "status"] = status
    df_estado.loc[df_estado["data"] == str(data), "ultima_execucao"] = datetime.now()
    df_estado.loc[df_estado["data"] == str(data), "erro"] = erro

    df_estado.to_csv("gerenciamento/log_processamento.csv", index=False)

def verificar_erros(df, var):
    erros_lista = []
    df = df[["accessed_date",var]]
    for idx, row in df.iterrows():
        motivo = []

        # Erro de data invalida
        if pd.isna(row["accessed_date"]):
            motivo.append("data_invalida")

        # Erro de valores nulos
        if row.isnull().any():
            motivo.append("valor_nulo")
            
        if motivo:
            linha_erro = row.copy()
            linha_erro["motivo_erro"] = "; ".join(motivo)
            erros_lista.append(linha_erro)

    df_erros = pd.DataFrame(erros_lista)

    return df_erros

# Vistas de lote - Parte 2
def acessar_dados(data, var):
    # Acessa os dados brutos separados por data
    df = pd.read_csv(f"dados_brutos/{data}.csv")
    
    # Salvando os arquivos de gerenciamento
    erros = verificar_erros(df, var)
    salvar_quarentena(erros, data)
    df = df.drop(erros.index)
    
    try:
        if df.empty:
            atualizar_estado(data, "erro", f"{data} sem dados válidos para {var}")
            return
        resultado = []
        
        # Calculando todas as métricas
        resultado = pd.DataFrame({
            'data': data,
            'var': var,
            'media': [df[var].mean().round(2)],
            'contagem': [df[var].count().round(2)],
            'maximo': [df[var].max().round(2)],
            'minimo': [df[var].min().round(2)],
            'mediana': [df[var].median().round(2)],
            'desvio_padrao': [df[var].std().round(2)],
            'variancia': [df[var].var().round(2)],
            'moda': [df[var].mode()[0].round(2)]
        })

        # Verifica se o arquivo existe, se não cria ele com a coluna fazer dinamicamente
        if not os.path.exists("vistas_lote/vistas_de_lote.csv"):
            vista_lote = resultado
        else:
            vista_lote = pd.read_csv('vistas_lote/vistas_de_lote.csv')

            # Cria a máscara para encontrar a linha específica
            mascara = ((vista_lote['data'] == str(data)) & (vista_lote['var'] == var))

            if mascara.any():
                vista_lote.loc[mascara, resultado.columns] = resultado.iloc[0].values
            else:
                vista_lote = pd.concat([vista_lote, resultado], ignore_index=True)

        # Salva o arquivo final e altera o log
        vista_lote.to_csv("vistas_lote/vistas_de_lote.csv", index=False)
        atualizar_estado(data, "processado")
        
    except Exception as e:
        print(f"Erro ao processar {data}: {e}")
        atualizar_estado(data, "erro")


def salvar_quarentena(df_erros, data):
    if df_erros.empty:
        return

    # Salva linhas problemáticas
    df_erros.to_csv(
        "gerenciamento/quarentena_linhas_brutas.csv",
        mode='a',
        index=False,
        header=not os.path.exists("gerenciamento/quarentena_linhas_brutas.csv")
    )

    # Salva o arquivo quarentena_linha_por_arquivo
    quarentena_linha_por_arquivo = pd.DataFrame({
        "data": [data],
        "quantidade_erros": [len(df_erros)]
    })

    # Salva o arquivo quarentena_linha_por_arquivo
    quarentena_linha_por_arquivo.to_csv(
        "gerenciamento/quarentena_linhas_por_arquivo.csv",
        mode='a',
        index=False,
        header=not os.path.exists("gerenciamento/quarentena_linhas_por_arquivo.csv")
    )


# Aplicando a <span style="color: #0077ff">**Camada de Lote**

In [6]:
# Inicializa o estado no log
inicializar_estado(df['data'].unique())

In [ ]:
# Gerando todos as vistas de lote das variáveis numéricas
for data in df['data'].unique():
        # Selecionamos apenas algumas variáveis pra não demorar pra rodar
        #for colum in ["duration_(secs)", "bytes", "age", "sales", "returned_amount"]:
        for colum in ["duration_(secs)","age"]:
                acessar_dados(data, colum)

# Funções <span style="color: #FF000A"> **Camada de Velocidade**

In [3]:
# Atualiza a vista rápida, calculando as métricas para cada linha
def atualizar_vista_rapida(linha, header, var):
    global estado_stream
    
    # Seta o path da contagem incremental
    caminho = "vistas_tempo_real/contagem_incremental.csv"
    
    # Separando os valores
    valores = [v.strip().replace('"', '') for v in linha.split(",")]

    try:
        # Pega o index da linha onde se encontra a var definida e puxa o valor
        idx = header.index(var)
        valor_str = valores[idx].strip()
        
        # Pula essa linha se tiver valores vazios ou nulos
        if not valor_str or valor_str == '' or valor_str == 'NA' or valor_str == 'null':
            return
            
        # Converte pra float e extrai o timestamp
        valor = float(valor_str)
        timestamp_completo = valores[0].strip()
        
        # Verifica se o timestamp é válido
        if not timestamp_completo:
            return
        
    # Pula silenciosamente linhas com erro
    except (ValueError, IndexError, AttributeError) as e:
        return

    # Verifica se tem a linha, caso nao tenha, ele cria com valores padrões
    if var not in estado_stream:
        estado_stream[var] = {"count": 0, "soma": 0, "max": valor, "min": valor, "valores": []}

    # Atualizando as métricas incrementaveis
    estado = estado_stream[var]
    estado["count"] += 1
    estado["soma"] += valor
    estado["max"] = max(estado["max"], valor)
    estado["min"] = min(estado["min"], valor)
    estado["valores"].append(valor)

    # Só calcula as métricas se tiver pelo menos 1 valor
    if estado["count"] == 0:
        return

    # Cálculo das métricas não incrementais
    resultados = {
        "media": estado["soma"] / estado["count"],
        "contagem": estado["count"],
        "maximo": estado["max"],
        "minimo": estado["min"],
        "mediana": pd.Series(estado["valores"]).median(),
        "desvio_padrao": pd.Series(estado["valores"]).std(),
        "variancia": pd.Series(estado["valores"]).var(),
        "moda": pd.Series(estado["valores"]).mode()[0] if not pd.Series(estado["valores"]).mode().empty else None
    }
    
    # Trava a thread para que nenhuma outra thread altere/acesse simultaneamente
    with lock_arquivo:

        # Criando a nova linha pra atualizar o arquivo contagem incremental
        nova_linha = {
            "data": timestamp_completo,
            "var": var,
            **resultados
        }

        # Se o arquivo ainda não existe, cria com a linha
        if not os.path.exists(caminho):
            df = pd.DataFrame([nova_linha])
        else:
            # Se já existir, abre o df
            df = pd.read_csv(caminho)

            # Se estiver vazio, coloca a nova linha
            if df.empty:
                df = pd.DataFrame([nova_linha])
            else:
                # Garantindo tipos corretos
                df["var"] = df["var"].astype(str)
                
                # Remove a linha antiga da mesma variável
                df = df[~((df["var"] == str(var)))]

                # Adiciona nova linha atualizada
                df = pd.concat([df, pd.DataFrame([nova_linha])], ignore_index=True)

        # Dropando linhas completamente vazias ou com NaN
        df = df.dropna(how='all')
        
        # Remove linhas onde 'var' é vazia
        if 'var' in df.columns:
            df = df[df['var'].astype(str).str.strip() != '']
            df = df[df['var'].notna()]

        # Arredondando as colunas para manter o padrão do arquivo original
        for col in ["media", "maximo", "minimo", "mediana", "desvio_padrao", "variancia", "moda"]:
            if col in df.columns:
                df[col] = pd.to_numeric(df[col], errors='coerce').round(2)
        # Sobrescreve o CSV inteiro
        df.to_csv(caminho, index=False)

# Função que reprocessa tudo chamando a função da camada de lote
def reconstruir_vistas_lote():
    print("Iniciando a reconstrução da vista de lote!")
    
    entrada = "vistas_tempo_real/contagem_incremental.csv"

    # Se não existir o arquivo de entrada não faz nada
    if not os.path.exists(entrada):
        print("Arquivo de entrada não encontrado!")
        return

    df_ent = pd.read_csv(entrada)

    # Se ele estiver vazio também não faz nada
    if df_ent.empty:
        print("Arquivo de entrada vazio!")
        return

    # Como a função original precisa de var e data, teremos que andar por todas as métricas recalculadas
    for i, row in df_ent.iterrows():
        data = row["data"].split(" ")[0]
        var = row['var']
        acessar_dados(data, var)
        
    print("Vista de lote atualizada com sucesso!")

def stream_dados(arq):
    global finaliza_laco
    while not finaliza_laco:
        linha = arq.readline().strip()
        if not linha:
            time.sleep(0.1)
            continue
        yield linha

def monitora_linhas(arquivo, vars):
    # Enquanto o arquivo log não existe ou ele não tem nada fica em stand by
    while not os.path.exists(arquivo) or os.path.getsize(arquivo) == 0:
        time.sleep(0.1)

    # Quando aparecer algo, chama a função de atualizar a vista rapida pra cada linha nova
    with open(arquivo, "r") as arq:
        header = [h.strip().replace('"', '') for h in arq.readline().strip().split(",")]
        for linha in stream_dados(arq):
            for colum in vars:
                atualizar_vista_rapida(linha, header, colum)

def simular_stream_csv(entrada, saida, delay=0.05):
    global finaliza_laco
    
    # Garantindo que diretório do fluxo.log exista
    os.makedirs(os.path.dirname(saida), exist_ok=True)
    
    # Pega a linha e escreve no arquivo fluxo.log
    with open(entrada, "r") as arq_in:
        with open(saida, "w") as arq_out:
            # Puxa o header e escreve no log
            header = arq_in.readline()
            arq_out.write(header)
            arq_out.flush()
            
            # Pra cada linha escrevemos ela no log
            for linha in arq_in:
                if finaliza_laco: break
                arq_out.write(linha)
                arq_out.flush()
                time.sleep(delay)

# Aplicando a <span style="color: #FF000A">**Camada de Velocidade**

In [6]:
finaliza_laco = False
estado_stream = {}
lock_arquivo = threading.Lock()

# Lista com todoas as métricas que calculamos
metricas = ["media","contagem","maximo","minimo","mediana","desvio_padrao","variancia","moda"]

if __name__ == "__main__":
    # Seta os paths de entrada, saida e stream
    arquivo_entrada = "dados_brutos/2017-03-21.csv"
    arquivo_stream = "dados_novos/fluxo.log"
    csv_tempo_real = "vistas_tempo_real/contagem_incremental.csv"
    
    # Esse campo controlará as variáveis que serão calculadas (selecionamos algumas para rodar mais rápido)
    # Todas colunas calculaveis: ["duration_(secs)", "bytes", "age", "sales", "returned_amount"]:
    vars = ["duration_(secs)","returned_amount"]

    # Garante que os diretórios existem
    os.makedirs("dados_novos", exist_ok=True)
    os.makedirs("vistas_tempo_real", exist_ok=True)

    # Se o arquivo contagem_incremental existir, reseta ele
    if os.path.exists(csv_tempo_real): 
        os.remove(csv_tempo_real)
    
    # Cria o arquivo de log se ele não existir
    open(arquivo_stream, "w").close()

    # Define as threads
    t1 = threading.Thread(target=simular_stream_csv, args=(arquivo_entrada, arquivo_stream), daemon=True)
    t2 = threading.Thread(target=monitora_linhas, args=(arquivo_stream, vars), daemon=True)

    # Starta as threads
    t1.start()
    t2.start()

    try:
        while t1.is_alive():
            time.sleep(1)
    # Só para quando tiver um Interrupt, ou seja, quando para a célula
    except KeyboardInterrupt:
        pass
    finally:
        # Finaliza as threads e chama a reconstruir_vistas_lote
        finaliza_laco = True
        t1.join()
        t2.join()
        reconstruir_vistas_lote()
        print("Processo finalizado!")

Iniciando a reconstrução da vista de lote!
Vista de lote atualizada com sucesso!
Processo finalizado!
